# Modelo 4: PIGNN con dos salidas, regresion de senal y clasificacion AAMI

Mismo pipeline que `Modelo3.ipynb` (MIT-BIH, ventanas por latido, clases AAMI,
pesos con tope, checkpoint, bitacora de test) pero con el codificador de
`Modelo2.ipynb`: el latido se propaga por el **grafo de conduccion cardiaca**.

El modelo tiene **dos cabezas sobre el mismo codificador PIGNN**:

| Salida | Lee | Predice |
|---|---|---|
| **Regresion** | `node_states` completo | la forma de onda, `[B, T, 1]` |
| **Clasificacion** | `z`, el latente de la atencion | la clase AAMI |

La regresion no lee `z` a proposito: la atencion colapsa el tiempo a un vector, y
con un vector no se reconstruye una onda. Cada cabeza tira del codificador en una
direccion distinta, y eso es el diseno, no un efecto colateral.

## La conexion con main.py

El latente `z` es ademas el punto donde se unen los tres sistemas del repo:

| Direccion | Que pasa |
|---|---|
| PIGNN -> `z` | atencion sobre nodos y luego sobre tiempo |
| `z` -> `main.py` | una cabeza auxiliar debe reproducir el veredicto del arbol |
| `main.py` -> `z` | el arbol suave modula `z` por FiLM antes de clasificar |
| lote -> `z` | prototipos por regimen, promediados sobre el lote actual |

> **Diferencia con `Modelo2.ipynb`:** no hay pseudo-PPG, asi que **no existe la
> fuga de datos** documentada ahi. El objetivo de regresion por defecto es el
> canal 1 del registro predicho desde el canal 0: derivaciones fisicamente
> distintas, y la objetivo nunca entra al modelo.


## Instalacion

El proyecto se maneja con `uv`. Desde la raiz del repo, en una terminal:

```powershell
uv sync
```

Y este notebook se abre con el kernel del entorno de `uv` (`uv run jupyter lab`).

## Imports

In [ ]:
from dataclasses import asdict
from heart_pignn.aami import CLASS_NAMES

from heart_pignn.data import DataConfig, MITBIHBeatDataset
from heart_pignn.explain import (
    contrast_against_normal,
    node_importance_by_class,
    print_node_report,
)
from heart_pignn.model import ModelConfig
from heart_pignn.train import (
    TrainConfig,
    build_datasets,
    build_loaders,
    build_model,
    compute_class_weights,
    run_test,
    run_training,
    summarize_rule_baseline,
)
from heart_pignn.utils import pick_device, set_seed

## Configuracion

Un solo objeto controla datos, modelo y entrenamiento. Cambiar aqui es lo
unico que hace falta para lanzar una ablacion.

In [ ]:
cfg = TrainConfig(
    data_root="mit-bih-arrhythmia-database-1.0.0",
    output_dir="checkpoints_pignn",
    epochs=30,             # epocas de ESTA sesion; se suman a lo que ya haya en el checkpoint
    batch_size=128,
    lr=1e-3,
    patience=10,
    class_weight_cap=6.0,  # mismo tope que corrigio el sobre-prediccion de S y F en Modelo3
    rule_w=0.3,            # peso de la cabeza auxiliar de reglas; 0 = sin conexion con main.py
    phys_w=0.05,           # regularizacion fisiologica heredada de Modelo2
    recon_w=0.5,           # peso de la cabeza de regresion; 0 la desactiva
    selection_metric="f1", # "combined" penaliza el PRD al elegir checkpoint
    samples_per_epoch=20000,
    num_workers=0,         # en Windows debe quedarse en 0
    seed=42,
    data=DataConfig(rr_context=10, augment=True, regression_target="cross_lead"),
    model=ModelConfig(hidden_dim=64, graph_steps=32, n_layers=2, dropout=0.1),
)

set_seed(cfg.seed)
device = pick_device(cfg.device)
print("device:", device)

## Datos

El split es **por registro**, no por latido. Dos ventanas del mismo paciente
comparten morfologia; repartirlas entre train y test infla las metricas sin que
nada en el codigo se vea sospechoso.

Si todavia no descargas MIT-BIH, puedes probar el pipeline completo con datos
sinteticos:

```python
from heart_pignn.demo_data import generate_dataset
generate_dataset("demo-mitdb", n_records=12, n_beats=300)
cfg.data_root = "demo-mitdb"
```

In [ ]:
datasets = build_datasets(cfg)
loaders = build_loaders(cfg, datasets)

## Linea base: que tanto sabe el arbol por si solo

Antes de entrenar conviene saber cuanto aporta `main.py` sin ayuda. Si el arbol
ya separara bien las clases AAMI, la red seria decorativa; si no separa nada,
cualquier ganancia posterior viene del PIGNN. Casi siempre queda en medio, y ese
"en medio" es el argumento de que la conexion sirve.

In [ ]:
agreement = summarize_rule_baseline(datasets['test'])

## Modelo

In [ ]:
model = build_model(cfg, device)
print()
print(model.config)

## Pesos de clase con tope

Sin tope, `F` puede pesar dos ordenes de magnitud mas que `N`, el modelo
sobre-predice la clase rara y la precision se desploma: es exactamente el fallo
documentado en la primera version de `Modelo3.ipynb`.

In [ ]:
weights = compute_class_weights(datasets['train'], cfg.class_weight_cap)

## Entrenamiento

`run_training` reanuda solo si ya existe `best_model.pt` en `output_dir`. Para
empezar de cero, borra esa carpeta.

En CPU cada epoca de 20 000 latidos tarda del orden de minutos; el bucle sobre
pasos de grafo es lo caro. Si vas lento, baja `graph_steps` a 16 o
`samples_per_epoch` a 5000 mientras exploras.

In [ ]:
result = run_training(cfg)

## Evaluacion en test, con comparacion contra la corrida anterior

Cada corrida se anexa a `checkpoints_pignn/test_runs.json`, igual que en `Modelo3.ipynb`.

In [ ]:
current = run_test(cfg, loaders=loaders, model=model)

## La salida de regresion

`run_test` ya imprime RMSE, PRD y Pearson. Aqui se ve la onda: prediccion contra
objetivo real para unos latidos del conjunto de test.

Lee el PRD con contexto. La referencia clinica habitual (<24%) viene de la
literatura de compresion de ECG, donde se reconstruye la *misma* derivacion. Esto
predice una derivacion distinta desde otra, que es mas dificil. Comparalo contra
la ablacion `recon_w=0` y contra `regression_target="reconstruct"`, no contra ese
umbral.

In [ ]:
import matplotlib.pyplot as plt
import torch

batch = next(iter(loaders["test"]))
model.eval()
with torch.no_grad():
    out = model(batch["x"].to(device), batch["rule_vec"].to(device), batch["regime"].to(device))

pred = out["signal"].cpu().numpy()
true = batch["y_signal"].numpy()
mask = batch["signal_mask"].numpy()
shown = [i for i in range(len(mask)) if mask[i] > 0][:4]

fig, axes = plt.subplots(len(shown), 1, figsize=(9, 2.2 * len(shown)), sharex=True)
for ax, i in zip(axes.ravel(), shown):
    ax.plot(true[i, :, 0], label="objetivo (canal 1)", linewidth=1.2)
    ax.plot(pred[i, :, 0], label="prediccion", linewidth=1.2, alpha=0.85)
    ax.set_ylabel(CLASS_NAMES[int(batch["y"][i])])
axes.ravel()[0].legend(loc="upper right", fontsize=8)
axes.ravel()[-1].set_xlabel("muestras (360 Hz)")
plt.tight_layout()
plt.show()

## Que mira la atencion

Aqui esta lo que el clasificador CNN de `Modelo3.ipynb` no podia dar. Su
atencion pesaba posiciones de un mapa convolucional, sin significado anatomico.
Esta pesa **nodos con nombre**, asi que se puede preguntar si un latido
clasificado como ventricular activo de verdad los nodos ventriculares.

Leelo con cuidado: que la atencion se concentre en un nodo es una explicacion
plausible, no una causa demostrada.

In [ ]:
importance = node_importance_by_class(model, loaders["test"], device, max_batches=20)
print_node_report(importance)
contrast_against_normal(importance, "V")
contrast_against_normal(importance, "S")

## Ablaciones para el reporte

Tres corridas que responden preguntas distintas. Cada una necesita su propio
`output_dir`, o reanudara desde el checkpoint equivocado.

| Corrida | Cambio | Pregunta que responde |
|---|---|---|
| sin reglas | `rule_w=0.0` | cuanto aporta conectar `main.py` al latente |
| sin fisica | `phys_w=0.0` | cuanto aporta la regularizacion heredada de `Modelo2` |
| sin regresion | `recon_w=0.0` | cuanto aporta la cabeza de senal a la clasificacion |
| sin prototipos | `ModelConfig(use_prototypes=False)` | cuanto aportan los prototipos de lote |

Y la comparacion que de verdad importa para el reporte: correr `Modelo3.ipynb`
con la misma semilla y el mismo split, y contrastar F1 macro. Si el PIGNN no le
gana a la CNN 1D, eso tambien es un resultado que vale la pena reportar.

In [ ]:
# Descomenta para lanzar la ablacion sin conexion con main.py
# cfg_sin_reglas = TrainConfig(**{**asdict(cfg), "rule_w": 0.0,
#                                 "output_dir": "checkpoints_pignn_sin_reglas"})
# run_training(cfg_sin_reglas)
# run_test(cfg_sin_reglas)

## Limitaciones

- Las posiciones R vienen del archivo `.atr`, igual que en `Modelo3.ipynb` y en
  casi toda la literatura sobre MIT-BIH. Eso supone resuelto el problema de
  deteccion. `DataConfig(rr_source="detected")` mide el pipeline sin ese
  supuesto, y el numero baja.
- Los rasgos RR locales (`rr_prev_ratio`, `rr_next_ratio`) son fuertes por si
  solos: un latido prematuro se delata en el timing antes que en la forma. Para
  saber cuanto aporta la morfologia, corre con
  `DataConfig(use_local_rr=False)`.
- La hipoxemia esta desactivada. `main.py` la simulaba a partir de la proporcion
  de latidos anormales del registro, calculada con los simbolos del `.atr`, es
  decir con la etiqueta. Usarla aqui seria fuga directa.
- Un solo split y una sola semilla no dan una metrica creible. Corre 3 a 5
  semillas y reporta media y desviacion antes de poner un numero en el reporte.